# 14 — CTD Reliable Reasoning: Qwen2.5-1.5B Full Baselines + Graph Oracle

Full larger-model replication of Experiment 13 with **Base / Vanilla / Robust / Abstain / Graph Oracle**, three entity-disjoint splits, three seeds, and automatic CTD acquisition. This version also fixes CTD header parsing: CTD bulk files keep the header in a commented line, so the loader explicitly recovers that header instead of letting pandas treat the first data row as column names.


In [ ]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes sentencepiece requests
import os,re,gc,json,random,subprocess,sys,gzip,shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, requests
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,set_seed
from peft import LoraConfig,prepare_model_for_kbit_training
from trl import SFTConfig,SFTTrainer
MODEL_NAME='Qwen/Qwen2.5-1.5B-Instruct'; SEEDS=[1,2,3]; SPLITS=['ChemicalID','GeneID','DiseaseID']
N_TRAIN=1500; N_EVAL=100; MAX_STEPS=80
ROOT=Path('/content') if Path('/content').exists() else Path.cwd(); DATA_DIR=ROOT/'ctd_data'; DATA_DIR.mkdir(parents=True,exist_ok=True)
RESULT_DIR=Path('results/14'); RESULT_DIR.mkdir(parents=True,exist_ok=True)
print('Model:',MODEL_NAME); print('CUDA:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## Data acquisition
The notebook first reuses local CTD files, then tries CTD bulk download URLs, and only asks for manual upload if all automatic routes fail.


In [ ]:
CHEM_NAME='CTD_chem_gene_ixns.tsv.gz'; GD_NAMES=['CTD_curated_genes_diseases.tsv.gz','CTD_genes_diseases.tsv.gz']
def valid_gzip(path,min_bytes=10000):
    path=Path(path)
    if not path.exists() or path.stat().st_size<min_bytes:return False
    try:
        with open(path,'rb') as f:
            if f.read(2)!=b'\x1f\x8b':return False
        with gzip.open(path,'rb') as f:f.read(256)
        return True
    except Exception:return False
def find_local(name):
    for p in [Path.cwd()/name,ROOT/name,DATA_DIR/name,Path('/content/drive/MyDrive')/name,Path('/content/drive/MyDrive/ctd')/name,Path('/content/drive/MyDrive/data')/name]:
        if valid_gzip(p):print('Found:',p);return p
    return None
def download_ctd(name):
    dest=DATA_DIR/name
    urls=[f'https://ctdbase.org/reports/{name}',f'https://ctdbase.org/downloads/{name}',f'http://ctdbase.org/reports/{name}']
    for url in urls:
        try:
            print('Trying:',url)
            with requests.get(url,stream=True,timeout=(20,300),allow_redirects=True,headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(dest,'wb') as f:
                    for ch in r.iter_content(1024*1024):
                        if ch:f.write(ch)
            if valid_gzip(dest):print('Downloaded:',dest);return dest
        except Exception as e:print(' failed:',type(e).__name__,str(e)[:120])
        dest.unlink(missing_ok=True)
    return None
def ensure_ctd(names):
    if isinstance(names,str):names=[names]
    for n in names:
        p=find_local(n)
        if p:return p
    for n in names:
        p=download_ctd(n)
        if p:return p
    try:
        from google.colab import files
        print('Automatic download failed. Upload one of:',names); up=files.upload()
        for n in names:
            if n in up:
                p=DATA_DIR/n;p.write_bytes(up[n])
                if valid_gzip(p):return p
    except Exception:pass
    raise FileNotFoundError('Could not obtain CTD data: '+', '.join(names))
CHEM_GENE=ensure_ctd(CHEM_NAME); GENE_DISEASE=ensure_ctd(GD_NAMES)
print(CHEM_GENE); print(GENE_DISEASE)


## Robust CTD parser
CTD bulk TSV files place the real header in a commented line such as `# ChemicalName ...`. Using `comment='#'` with default `header=0` therefore makes pandas use the first biological record as the header. The parser below explicitly recovers the last tab-delimited comment line containing expected field names, then reads data with `header=None`.


In [ ]:
def read_ctd(path,expected_any):
    header=None
    with gzip.open(path,'rt',encoding='utf-8',errors='replace') as f:
        for line in f:
            if not line.startswith('#'):break
            s=line.lstrip('#').strip()
            if '\t' in s:
                cols=[x.strip() for x in s.split('\t')]
                if any(x in cols for x in expected_any):header=cols
    if header is None:
        raise ValueError(f'Could not recover CTD header from {path}. Expected one of {expected_any}')
    df=pd.read_csv(path,sep='\t',comment='#',compression='gzip',dtype=str,low_memory=False,header=None,names=header)
    print(Path(path).name,'columns:',df.columns.tolist()[:20]); return df
cg=read_ctd(CHEM_GENE,['ChemicalName','ChemicalID','GeneSymbol','GeneID'])
gd=read_ctd(GENE_DISEASE,['GeneSymbol','GeneID','DiseaseName','DiseaseID'])
def pick(df,names):
    for n in names:
        if n in df.columns:return n
    raise KeyError(f'None of {names} found. Columns={list(df.columns)[:30]}')
c_name=pick(cg,['ChemicalName']);c_id=pick(cg,['ChemicalID']);g_sym1=pick(cg,['GeneSymbol']);g_id1=pick(cg,['GeneID'])
g_sym2=pick(gd,['GeneSymbol']);g_id2=pick(gd,['GeneID']);d_name=pick(gd,['DiseaseName']);d_id=pick(gd,['DiseaseID'])
cg2=cg[[c_name,c_id,g_sym1,g_id1]].dropna().drop_duplicates();gd2=gd[[g_sym2,g_id2,d_name,d_id]].dropna().drop_duplicates()
cg2.columns=['ChemicalName','ChemicalID','GeneSymbol','GeneID'];gd2.columns=['GeneSymbol','GeneID','DiseaseName','DiseaseID']
paths=cg2.merge(gd2,on=['GeneSymbol','GeneID'],how='inner').drop_duplicates()
paths=paths[(paths.ChemicalName.str.len()<100)&(paths.DiseaseName.str.len()<120)].reset_index(drop=True)
assert len(paths)>3000,f'Too few joined paths: {len(paths)}'
print('Two-hop paths:',len(paths));display(paths.head())


In [ ]:
edge_pool=gd2[['GeneSymbol','DiseaseName']].drop_duplicates().reset_index(drop=True)
def render_prompt(row,edges):
    lines=[f'- {g} -> {d}' for g,d in edges]
    return 'Use only the supplied evidence. Determine the disease supported by the path from the queried chemical through the queried gene. If no supplied gene-disease relation supports the queried gene, answer exactly: No supported path.\n\n'+f'Chemical: {row.ChemicalName}\nGene: {row.GeneSymbol}\nEvidence:\n'+'\n'.join(lines)
def positive_edges(row,k,rng):
    edges=[(str(row.GeneSymbol),str(row.DiseaseName))];pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    if k:
        sub=pool.sample(n=k,random_state=rng.randint(0,2**31-1));edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges);return edges
def no_path_edges(row,k,rng,lexical=False):
    pool=edge_pool[(edge_pool.GeneSymbol!=row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)].copy();edges=[]
    if lexical:
        sym=str(row.GeneSymbol);near=pool[pool.GeneSymbol.astype(str).str.startswith(sym[:max(1,min(2,len(sym)))])]
        if len(near):
            x=near.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0];edges.append((str(x.GeneSymbol),str(x.DiseaseName)));pool=pool[pool.GeneSymbol!=x.GeneSymbol]
    need=k-len(edges)
    if need>0:
        sub=pool.sample(need,random_state=rng.randint(0,2**31-1));edges += [(str(g),str(d)) for g,d in sub.itertuples(index=False,name=None)]
    rng.shuffle(edges);return edges
def counterfactual_edges(row,rng):
    c=edge_pool[(edge_pool.GeneSymbol==row.GeneSymbol)&(edge_pool.DiseaseName!=row.DiseaseName)]
    cf=str(c.sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName) if len(c) else str(edge_pool[edge_pool.DiseaseName!=row.DiseaseName].sample(1,random_state=rng.randint(0,2**31-1)).iloc[0].DiseaseName)
    return [(str(row.GeneSymbol),cf)],cf
def answer_text(row):return f'Disease: {row.DiseaseName}. Reasoning: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}.'
def make_split(df,col,seed):
    r=np.random.default_rng(seed);ents=df[col].dropna().unique().copy();r.shuffle(ents);cut=max(1,int(.8*len(ents)));tr_e,te_e=set(ents[:cut]),set(ents[cut:])
    trp=df[df[col].isin(tr_e)];tep=df[df[col].isin(te_e)].drop_duplicates(['ChemicalID','GeneID','DiseaseID'])
    assert len(trp)>=N_TRAIN and len(tep)>=N_EVAL
    tr=trp.sample(N_TRAIN,random_state=seed).reset_index(drop=True);te=tep.sample(N_EVAL,random_state=1000+seed).reset_index(drop=True);assert set(tr[col]).isdisjoint(set(te[col]));return tr,te
def make_train_dataset(df,condition,seed):
    rng=random.Random(seed);rec=[]
    for _,row in df.iterrows():
        u=rng.random()
        if condition=='vanilla':edges=positive_edges(row,0,rng);ans=answer_text(row)
        elif condition=='robust':edges=positive_edges(row,rng.choice([1,3,5,10]),rng);ans=answer_text(row)
        else:
            if u<.34:edges=no_path_edges(row,rng.choice([1,3,5,10]),rng,rng.random()<.5);ans='No supported path.'
            else:edges=positive_edges(row,0 if u<.5 else rng.choice([1,3,5,10]),rng);ans=answer_text(row)
        rec.append({'text':render_prompt(row,edges)+'\nAnswer: '+ans})
    return Dataset.from_list(rec)
def item(row,edges,target,typ):return {'target_gene':str(row.GeneSymbol),'target_disease':None if target is None else str(target),'evidence_edges':[(str(g),str(d)) for g,d in edges],'prompt':render_prompt(row,edges),'answer_type':typ}
def make_eval_sets(df,seed):
    rng=random.Random(20000+seed);out={k:[] for k in ['clean','distractor_5','hard_no_path','lexical_no_path','counterfactual']}
    for _,row in df.iterrows():
        out['clean'].append(item(row,positive_edges(row,0,rng),row.DiseaseName,'positive'));out['distractor_5'].append(item(row,positive_edges(row,5,rng),row.DiseaseName,'positive'))
        out['hard_no_path'].append(item(row,no_path_edges(row,5,rng,False),None,'no_path'));out['lexical_no_path'].append(item(row,no_path_edges(row,5,rng,True),None,'no_path'))
        e,cf=counterfactual_edges(row,rng);out['counterfactual'].append(item(row,e,cf,'positive'))
    return out


## Graph Oracle + model evaluation


In [ ]:
def norm(s):return re.sub(r'\s+',' ',str(s).strip().lower())
def score_one(x,p):
    p=norm(p)
    if x['answer_type']=='no_path':return 'no supported path' in p
    return norm(x['target_disease']) in p and 'no supported path' not in p
def score_set(items,preds):return float(np.mean([score_one(x,p) for x,p in zip(items,preds)]))
def oracle_pred(x):
    ds=list(dict.fromkeys([d for g,d in x['evidence_edges'] if g==x['target_gene']]))
    if not ds:return 'No supported path.'
    if len(ds)==1:return 'Disease: '+ds[0]
    return 'AMBIGUOUS: '+' | '.join(ds)
def eval_oracle(es):
    sc={}
    for k,it in es.items():
        pr=[oracle_pred(x) for x in it];sc[k]=score_set(it,pr);print('ORACLE',k,sc[k])
    assert all(v==1.0 for v in sc.values()),f'Oracle failure: {sc}'
    return sc
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,use_fast=True);tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token;tokenizer.padding_side='left'
compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=compute_dtype,bnb_4bit_use_double_quant=True)
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
def load_model(training=False):
    m=AutoModelForCausalLM.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map='auto');m.config.use_cache=not training
    return prepare_model_for_kbit_training(m) if training else m
def train_model(ds,outdir,seed):
    set_seed(seed);m=load_model(True)
    args=SFTConfig(output_dir=outdir,dataset_text_field='text',max_length=768,per_device_train_batch_size=2,gradient_accumulation_steps=4,max_steps=MAX_STEPS,learning_rate=2e-4,warmup_ratio=.05,logging_steps=20,save_strategy='no',report_to='none',packing=False,gradient_checkpointing=True,bf16=compute_dtype==torch.bfloat16,fp16=compute_dtype==torch.float16)
    tr=SFTTrainer(model=m,args=args,train_dataset=ds,processing_class=tokenizer,peft_config=lora);tr.train();tr.model.config.use_cache=True;return tr.model
@torch.inference_mode()
def generate_batched(model,prompts,batch_size=8,max_new_tokens=64):
    model.eval();outs=[]
    for i in range(0,len(prompts),batch_size):
        pp=prompts[i:i+batch_size];chats=[[{'role':'user','content':p}] for p in pp];txt=[tokenizer.apply_chat_template(c,tokenize=False,add_generation_prompt=True) for c in chats];enc=tokenizer(txt,return_tensors='pt',padding=True,truncation=True,max_length=768).to(model.device)
        y=model.generate(**enc,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id);lens=enc['attention_mask'].sum(1).tolist()
        for j,z in enumerate(y):outs.append(tokenizer.decode(z[-max_new_tokens:],skip_special_tokens=True))
    return outs
def evaluate(model,es):
    return {k:score_set(v,generate_batched(model,[x['prompt'] for x in v])) for k,v in es.items()}


## Run full matrix and save/resume results


In [ ]:
RESULT_FILE=RESULT_DIR/'14_results.csv';SUMMARY_FILE=RESULT_DIR/'14_summary.csv';CONFIG_FILE=RESULT_DIR/'14_config.json'
def save(rows):
    df=pd.DataFrame(rows);df.to_csv(RESULT_FILE,index=False)
    sm=df.groupby(['split','condition','metric']).score.agg(['mean','std','count']).reset_index() if len(df) else pd.DataFrame();sm.to_csv(SUMMARY_FILE,index=False)
    CONFIG_FILE.write_text(json.dumps({'model':MODEL_NAME,'seeds':SEEDS,'splits':SPLITS,'n_train':N_TRAIN,'n_eval':N_EVAL,'max_steps':MAX_STEPS},indent=2));return df,sm
rows=pd.read_csv(RESULT_FILE).to_dict('records') if RESULT_FILE.exists() else [];done={(r['split'],int(r['seed']),r['condition']) for r in rows}
for split in SPLITS:
  for seed in SEEDS:
    print('\n===',split,'seed',seed,'===');tr_df,te_df=make_split(paths,split,seed);es=make_eval_sets(te_df,seed)
    key=(split,seed,'oracle')
    if key not in done:
        sc=eval_oracle(es)
        for m,s in sc.items():rows.append({'split':split,'seed':seed,'condition':'oracle','metric':m,'score':s,'n_eval':len(es[m])})
        save(rows);done.add(key)
    key=(split,seed,'base')
    if key not in done:
        m=load_model(False);sc=evaluate(m,es)
        for met,s in sc.items():rows.append({'split':split,'seed':seed,'condition':'base','metric':met,'score':s,'n_eval':len(es[met])})
        del m;gc.collect();torch.cuda.empty_cache();save(rows);done.add(key);print('base',sc)
    for cond in ['vanilla','robust','abstain']:
        key=(split,seed,cond)
        if key in done:continue
        ds=make_train_dataset(tr_df,cond,seed);m=train_model(ds,str(RESULT_DIR/f'{split}_{seed}_{cond}'),seed);sc=evaluate(m,es)
        for met,s in sc.items():rows.append({'split':split,'seed':seed,'condition':cond,'metric':met,'score':s,'n_eval':len(es[met])})
        del m,ds;gc.collect();torch.cuda.empty_cache();save(rows);done.add(key);print(cond,sc)
df,summary=save(rows);display(summary)
